<a href="https://colab.research.google.com/github/genairsh/GenAI_Level3_Assignment/blob/main/bronze/Policy%26Claims_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.5 MB/s eta 0:00:00


In [2]:
import pymupdf
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

In [4]:
#Load the PDF
import pymupdf
import requests
import io

pdf_url = "https://raw.githubusercontent.com/genairsh/GenAI_Level3_Assignment/main/datasets/health_insurance_policy.pdf"

# Download the PDF content
response = requests.get(pdf_url)
response.raise_for_status() # Raise an exception for bad status codes

# Open the PDF from the in-memory bytes
doc = pymupdf.open(stream=io.BytesIO(response.content), filetype="pdf")

# Store text along with page number
pages = []
for page_number, page in enumerate(doc, start=1):
    text = page.get_text()
    pages.append({
        "page": page_number,
        "text": text
    })

print("Number of pages:", len(pages))

Number of pages: 4


In [5]:
for page in pages[:2]:

    print("PAGE:", page["page"])
    print("-" * 50)
    print(page["text"][:1000])
    print()

PAGE: 1
--------------------------------------------------
Page 1
SAMPLE HEALTH INSURANCE POLICY
Dummy Policy Document for Policy & Claims Copilot RAG Project
Important: This is a completely fictional policy created for educational, RAG, LLM, testing, and demonstration purposes. It is
not a real insurance contract and must not be used for an actual insurance claim or financial decision.
Policy ID
HIC-2026-001
Policy Name
SecureCare Family Health Plan
Policy Version
1.0
Effective Date
01 January 2026
Policy Type
Individual and Family Health Insurance
Annual Sum Insured
I10,00,000
Geographical Coverage
India
1. Eligibility and Policy Period
The SecureCare Family Health Plan is available to persons aged 18 to 65 years at policy inception. Dependent
children may be covered from 91 days to 25 years of age when included in the family policy. The standard
policy period is one year from the effective date shown in the policy schedule.
2. Hospitalization Coverage
The policy covers medically nec

In [6]:
#remove completely empty pages/text
pages = [
    page for page in pages
    if page["text"].strip()
]

print("Pages containing text:", len(pages))

Pages containing text: 4


In [7]:
for page in pages:
    print("Page:", page["page"])
    print("Characters:", len(page["text"]))
    print()

Page: 1
Characters: 2684

Page: 2
Characters: 2822

Page: 3
Characters: 659

Page: 4
Characters: 824



In [8]:
import re

def clean_text(text):
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove spaces around punctuation
    text = re.sub(r'\s+([.,:;])', r'\1', text)

    return text.strip()


def detect_section(text, current_section="General"):
    """
    Try to identify a section heading from the text.
    """

    lines = text.split("\n")

    for line in lines:
        line = line.strip()

        # Ignore very short lines
        if len(line) < 3:
            continue

        # Detect headings such as:
        # 1. Hospitalization Coverage
        # 2. Waiting Periods
        # 10. Exclusions

        if re.match(r'^\d+[\.\)]\s+', line):
            return line

        # Detect lines that look like headings
        if (
            len(line) < 80
            and line.isupper()
            and not line.endswith(".")
        ):
            return line

    return current_section


def create_robust_chunks(pages, chunk_size=800, overlap=120):

    chunks = []

    current_section = "General"

    for page_data in pages:

        page_number = page_data["page"]
        raw_text = page_data["text"]

        # Split page into lines first
        lines = raw_text.split("\n")

        current_section = detect_section(
            raw_text,
            current_section
        )

        cleaned_lines = []

        for line in lines:
            line = line.strip()

            if line:
                cleaned_lines.append(line)

        page_text = " ".join(cleaned_lines)

        page_text = clean_text(page_text)

        start = 0

        while start < len(page_text):

            end = min(start + chunk_size, len(page_text))

            chunk_text = page_text[start:end]

            # Try to avoid cutting in the middle of a sentence
            if end < len(page_text):

                last_period = chunk_text.rfind(".")
                last_question = chunk_text.rfind("?")
                last_colon = chunk_text.rfind(":")

                best_break = max(
                    last_period,
                    last_question,
                    last_colon
                )

                if best_break > chunk_size * 0.5:
                    end = start + best_break + 1
                    chunk_text = page_text[start:end]

            chunk_text = chunk_text.strip()

            if chunk_text:

                chunks.append({
                    "text": chunk_text,
                    "page": page_number,
                    "section": current_section
                })

            new_start = end - overlap

            # Safety check to avoid infinite loops
            if new_start <= start:
                new_start = end

            start = new_start

    return chunks

In [9]:
chunks = create_robust_chunks(
    pages,
    chunk_size=800,
    overlap=120
)

print("Total number of chunks:", len(chunks))

Total number of chunks: 17


In [10]:
###Inspect the chunks and metadata
for i, chunk in enumerate(chunks[:10]):

    print("=" * 80)
    print("CHUNK:", i)
    print("PAGE:", chunk["page"])
    print("SECTION:", chunk["section"])
    print("=" * 80)
    print(chunk["text"])
    print()

CHUNK: 0
PAGE: 1
SECTION: SAMPLE HEALTH INSURANCE POLICY
Page 1 SAMPLE HEALTH INSURANCE POLICY Dummy Policy Document for Policy & Claims Copilot RAG Project Important: This is a completely fictional policy created for educational, RAG, LLM, testing, and demonstration purposes. It is not a real insurance contract and must not be used for an actual insurance claim or financial decision. Policy ID HIC-2026-001 Policy Name SecureCare Family Health Plan Policy Version 1.0 Effective Date 01 January 2026 Policy Type Individual and Family Health Insurance Annual Sum Insured I10,00,000 Geographical Coverage India 1. Eligibility and Policy Period The SecureCare Family Health Plan is available to persons aged 18 to 65 years at policy inception. Dependent children may be covered from 91 days to 25 years of age when included in the family policy.

CHUNK: 1
PAGE: 1
SECTION: SAMPLE HEALTH INSURANCE POLICY
policy inception. Dependent children may be covered from 91 days to 25 years of age when include

In [11]:
#Create Normalized Embedding
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (17, 384)


In [13]:
#Create FAISS Vector Database
dimension = embeddings.shape[1] # extracts the 2nd element of this tuple.i.e 384

index = faiss.IndexFlatIP(dimension) ###Cosine Similarity Search

index.add(embeddings) # add embeddings to the FAISS index.

print("Number of vectors stored:", index.ntotal)

Number of vectors stored: 17


In [14]:
###Retrieval Threshold
def retrieve_documents(
    question,
    k=5,
    similarity_threshold=0.30
):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_embedding = question_embedding.astype("float32")

    scores, indices = index.search(
        question_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        if score >= similarity_threshold:

            results.append({
                "text": chunks[idx]["text"],
                "page": chunks[idx]["page"],
                "section": chunks[idx]["section"],
                "score": float(score)
            })

    return results

In [17]:
question = "What is the room rent limit?"

results = retrieve_documents(
    question,
    k=5
)

for i, result in enumerate(results, start=1):

    print("=" * 80)
    print("RESULT:", i)
    print("PAGE:", result["page"])
    print("SECTION:", result["section"])
    print("SIMILARITY:", round(result["score"], 3))
    print("=" * 80)
    print(result["text"])
    print()

RESULT: 1
PAGE: 1
SECTION: SAMPLE HEALTH INSURANCE POLICY
SIMILARITY: 0.421
om Rent and Nursing Charges Room rent and nursing charges are covered up to I5,000 per day for standard hospitalization. Eligible ICU room charges are covered up to I10,000 per day. If the selected room exceeds the applicable limit, payment may be restricted according to the eligible limit and applicable proportionate deductions. 6. Co-payment A 10% co-payment applies to eligible hospitalization expenses for insured persons who enter the policy at age 61 or above. For insured persons below age 61, no standard age-based co-payment applies unless an endorsement states otherwise. 7. Waiting Periods General: 30 days from policy start for illnesses, except accidental injury. Specified diseases: 24 months for selected diseases and procedures listed in the policy schedule. Pre-existing diseases:

RESULT: 2
PAGE: 1
SECTION: SAMPLE HEALTH INSURANCE POLICY
SIMILARITY: 0.379
t to limits and exclusions. Hospitalization mus

In [20]:
def build_context(results):
    context_parts = []
    for i, result in enumerate(results, start=1):

        context_parts.append(
            f"""
SOURCE {i}
Page: {result['page']}
Section: {result['section']}
Similarity: {result['score']:.3f}

Policy Text:
{result['text']}
"""
        )
    return "\n".join(context_parts)

In [21]:
###Build the context
context = build_context(results)

print(context)


SOURCE 1
Page: 1
Section: SAMPLE HEALTH INSURANCE POLICY
Similarity: 0.421

Policy Text:
om Rent and Nursing Charges Room rent and nursing charges are covered up to I5,000 per day for standard hospitalization. Eligible ICU room charges are covered up to I10,000 per day. If the selected room exceeds the applicable limit, payment may be restricted according to the eligible limit and applicable proportionate deductions. 6. Co-payment A 10% co-payment applies to eligible hospitalization expenses for insured persons who enter the policy at age 61 or above. For insured persons below age 61, no standard age-based co-payment applies unless an endorsement states otherwise. 7. Waiting Periods General: 30 days from policy start for illnesses, except accidental injury. Specified diseases: 24 months for selected diseases and procedures listed in the policy schedule. Pre-existing diseases:


SOURCE 2
Page: 1
Section: SAMPLE HEALTH INSURANCE POLICY
Similarity: 0.379

Policy Text:
t to limits and exc

In [22]:
def create_rag_prompt(question, context):

    prompt = f"""
You are Policy & Claims Copilot.

Your job is to answer questions ONLY using the supplied
insurance policy context.

IMPORTANT GROUNDING RULES:

1. Use ONLY information explicitly present in the policy context.

2. NEVER invent:
   - coverage
   - limits
   - waiting periods
   - exclusions
   - benefits
   - claim timelines
   - documents

3. Do NOT use general insurance knowledge.

4. If the policy context does not contain enough information
   to answer the question, DO NOT guess.

5. In that situation, respond exactly in this style:

   "I cannot confirm this from the available policy information."

6. Every answer must include the relevant source page
   and section.

7. If multiple policy sections are relevant, mention all
   relevant pages and sections.

8. Clearly distinguish between:
   - confirmed policy information
   - information that cannot be confirmed

9. Do not approve or reject an insurance claim.

10. For claims, provide only a preliminary pre-check.

POLICY CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    return prompt

In [23]:
###Invoke the prompt
prompt = create_rag_prompt(
    question,
    context
)

print(prompt)


You are Policy & Claims Copilot.

Your job is to answer questions ONLY using the supplied
insurance policy context.

IMPORTANT GROUNDING RULES:

1. Use ONLY information explicitly present in the policy context.

2. NEVER invent:
   - coverage
   - limits
   - waiting periods
   - exclusions
   - benefits
   - claim timelines
   - documents

3. Do NOT use general insurance knowledge.

4. If the policy context does not contain enough information
   to answer the question, DO NOT guess.

5. In that situation, respond exactly in this style:

   "I cannot confirm this from the available policy information."

6. Every answer must include the relevant source page
   and section.

7. If multiple policy sections are relevant, mention all
   relevant pages and sections.

8. Clearly distinguish between:
   - confirmed policy information
   - information that cannot be confirmed

9. Do not approve or reject an insurance claim.

10. For claims, provide only a preliminary pre-check.

POLICY CONTEXT

In [24]:
!pip install -q google-genai

In [25]:
from google import genai
from google.colab import userdata

In [26]:
api_key = userdata.get("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [51]:

def call_llm(prompt):
   # MODEL_NAME = os.getenv("MODEL_NAME","gemini-3.6-flash")
    MODEL_NAME = userdata.get("MODEL_NAME")
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text

In [45]:
def ask_policy_question(question):

    results = retrieve_documents(
        question,
        k=5
    )

    if len(results) == 0:

        return {
            "answer": "I cannot confirm this from the available policy information.",
            "sources": []
        }

    context = build_context(results)

    prompt = create_rag_prompt(
        question,
        context
    )

    answer = call_llm(prompt)

    sources = []

    for result in results:

        source = {
            "page": result["page"],
            "section": result["section"],
            "score": result["score"]
        }

        sources.append(source)

    return {
        "answer": answer,
        "sources": sources
    }

In [46]:
def display_policy_answer(question):

    response = ask_policy_question(question)

    print("=" * 80)
    print("POLICY & CLAIMS COPILOT")
    print("=" * 80)

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(response["answer"])

    print("\nSOURCE PAGES / SECTIONS:")

    if response["sources"]:

        seen = set()

        for source in response["sources"]:

            key = (
                source["page"],
                source["section"]
            )

            if key not in seen:

                print(
                    f"- Page {source['page']} | "
                    f"{source['section']}"
                )

                seen.add(key)

    else:

        print("- No supporting policy source found.")

    print("=" * 80)

In [52]:
display_policy_answer(
    "What is the room rent limit?"
)

POLICY & CLAIMS COPILOT

QUESTION:
What is the room rent limit?

ANSWER:
Based on the provided policy context, here is the confirmed policy information:

**Confirmed Policy Information:**
* **Standard Hospitalization Room Rent:** Room rent and nursing charges are covered up to I5,000 per day.
* **ICU Room Charges:** Eligible ICU room charges are covered up to I10,000 per day.
* **Proportionate Deductions:** If the selected room exceeds the applicable limit, payment may be restricted according to the eligible limit and applicable proportionate deductions.

**Sources:**
* **Page:** 1
* **Section:** SAMPLE HEALTH INSURANCE POLICY (Section 5: Room Rent and Nursing Charges)

SOURCE PAGES / SECTIONS:
- Page 1 | SAMPLE HEALTH INSURANCE POLICY


In [34]:
###Claims Precheck
def claims_precheck(claim_scenario):

    results = retrieve_documents(
        claim_scenario,
        k=6
    )

    if len(results) == 0:

        return {
            "precheck": (
                "Insufficient policy information available "
                "to perform the pre-check."
            ),
            "sources": []
        }

    context = build_context(results)

    prompt = f"""
You are an Insurance Claims Pre-check Assistant.

Your task is to perform a PRELIMINARY policy-based
claims assessment.

Use ONLY the supplied policy context.

Do NOT make a final claim decision.

Do NOT say:
- Claim approved
- Claim rejected
- Definitely payable
- Definitely not payable

Instead identify possible policy conditions.

Analyze the scenario under these headings:

1. Potentially Relevant Coverage
2. Waiting Period Check
3. Limits / Sub-limits
4. Possible Exclusions
5. Required Documents
6. Missing Information
7. Recommended Next Steps
8. Preliminary Assessment

For the Preliminary Assessment use one of:

- Potentially Covered — subject to verification
- Potentially Not Covered — policy condition may apply
- Insufficient Information — more details required

IMPORTANT:

If something is not stated in the supplied policy,
say:

"Not specified in the available policy information."

Do not guess.

Every important policy statement must mention
its source page and section.

POLICY CONTEXT:
{context}

CLAIM SCENARIO:
{claim_scenario}

PRE-CHECK:
"""

    answer = call_llm(prompt)

    sources = []

    for result in results:

        sources.append({
            "page": result["page"],
            "section": result["section"],
            "score": result["score"]
        })

    return {
        "precheck": answer,
        "sources": sources
    }

In [35]:
#Display Claims Pre-check
def display_claim_precheck(claim_scenario):

    response = claims_precheck(
        claim_scenario
    )

    print("=" * 80)
    print("CLAIMS PRE-CHECK")
    print("=" * 80)

    print("\nCLAIM SCENARIO:")
    print(claim_scenario)

    print("\nPRELIMINARY ASSESSMENT:")
    print(response["precheck"])

    print("\nSOURCE PAGES / SECTIONS:")

    seen = set()

    for source in response["sources"]:

        key = (
            source["page"],
            source["section"]
        )

        if key not in seen:

            print(
                f"- Page {source['page']} | "
                f"{source['section']}"
            )

            seen.add(key)

    print("\nNOTE:")
    print(
        "This is a preliminary policy-based pre-check "
        "and not a final claim decision."
    )

    print("=" * 80)

**Test Cases**

1. Coverage Question

In [74]:
display_policy_answer(
    "Is hospitalization covered under this policy?"
)

POLICY & CLAIMS COPILOT

QUESTION:
Is hospitalization covered under this policy?

ANSWER:
**Confirmed Policy Information:**

Yes, hospitalization is covered under this policy. 

According to the policy terms:
* The policy covers medically necessary hospitalization for illness or accidental injury when the insured person is admitted for more than 24 hours.
* Covered expenses may include room charges, nursing charges, doctor fees, prescribed medicines, diagnostic tests, and eligible hospital services.
* Coverage is subject to policy limits, exclusions, medical necessity, and must be supported by appropriate medical records.

**Source:**
* **Page:** 1
* **Section:** SAMPLE HEALTH INSURANCE POLICY (Section 2. Hospitalization Coverage)

SOURCE PAGES / SECTIONS:
- Page 1 | SAMPLE HEALTH INSURANCE POLICY
- Page 3 | 18. Policy Support Questions
- Page 2 | 8. Maternity Benefit


2. Limit Question

In [53]:
display_policy_answer(
    "What is the room rent limit?"
)

POLICY & CLAIMS COPILOT

QUESTION:
What is the room rent limit?

ANSWER:
Based on the provided policy context, the room rent limits are as follows:

* **Standard Hospitalization:** Room rent and nursing charges are covered up to **I5,000 per day**.
* **ICU Room Charges:** Eligible ICU room charges are covered up to **I10,000 per day**.

*(Note: If the selected room exceeds the applicable limit, payment may be restricted according to the eligible limit and applicable proportionate deductions.)*

**Source:** 
* Page 1, Section: SAMPLE HEALTH INSURANCE POLICY (5. Room Rent and Nursing Charges)

SOURCE PAGES / SECTIONS:
- Page 1 | SAMPLE HEALTH INSURANCE POLICY


3. Waiting-period question

In [ ]:
display_policy_answer(
    "What is the waiting period for pre-existing diseases?"
)

POLICY & CLAIMS COPILOT

QUESTION:
What is the waiting period for pre-existing diseases?

ANSWER:
**Confirmed Policy Information:**
The waiting period for pre-existing diseases is 36 months.

**Information That Cannot Be Confirmed:**
Any specific exceptions, conditions, or terms following "unless the..." cannot be confirmed from the available policy text, as the excerpt ends abruptly.

**Source Page & Section:**
* **Page:** 1
* **Section:** SAMPLE HEALTH INSURANCE POLICY (Source 1, Source 2, Source 4)

SOURCE PAGES / SECTIONS:
- Page 1 | SAMPLE HEALTH INSURANCE POLICY


4. Document Question

In [ ]:
display_policy_answer(
    "What documents are required for a hospitalization reimbursement claim?"
)

POLICY & CLAIMS COPILOT

QUESTION:
What documents are required for a hospitalization reimbursement claim?

ANSWER:
Based on the supplied policy context, here is the confirmed information regarding the required documents for a hospitalization reimbursement claim:

**Confirmed Policy Information:**
Required documents for a hospitalization claim include:
* Completed claim form
* Policy or health card details
* Identity proof (when requested)
* Admission and discharge summary
* Final hospital bill and detailed bill breakup
* Payment receipts
* Doctor prescriptions and consultation records
* Relevant diagnostic reports
* Pharmacy bills and prescriptions (where applicable)
* Treatment summary
* Bank account details for reimbursement (when required)

Additionally, during claim verification, the claims team may request:
* Previous medical records
* Original reports
* Treatment notes
* Itemized bills
* Clarification from the hospital

**Information That Cannot Be Confirmed:**
* Any additional s

5. Claim Timeline

In [ ]:
display_policy_answer(
    "How soon should emergency hospitalization be reported?"
)

POLICY & CLAIMS COPILOT

QUESTION:
How soon should emergency hospitalization be reported?

ANSWER:
**Confirmed Policy Information:**
For emergency hospitalization, you must notify the insurer or claims administrator within 24 hours of admission or as soon as reasonably possible.

**Sources:**
* Page 2, Section: 8. Maternity Benefit

SOURCE PAGES / SECTIONS:
- Page 2 | 8. Maternity Benefit
- Page 1 | SAMPLE HEALTH INSURANCE POLICY


6. Exclusion Question


In [ ]:
display_policy_answer(
    "Is cosmetic treatment covered?"
)

POLICY & CLAIMS COPILOT

QUESTION:
Is cosmetic treatment covered?

ANSWER:
**Confirmed Policy Information:**
Cosmetic or aesthetic treatment is excluded from coverage unless it is specifically endorsed, with an exception for eligible accident-related treatment.

**Source:**
- Page: 2, Section: 8. Maternity Benefit

SOURCE PAGES / SECTIONS:
- Page 2 | 8. Maternity Benefit
- Page 1 | SAMPLE HEALTH INSURANCE POLICY
- Page 3 | 18. Policy Support Questions


7. Refusal

In [ ]:
display_policy_answer(
    "Does this policy provide a ₹2,00,000 annual dental treatment benefit?"
)

POLICY & CLAIMS COPILOT

QUESTION:
Does this policy provide a ₹2,00,000 annual dental treatment benefit?

ANSWER:
I cannot confirm this from the available policy information.

* **Confirmed Policy Information:** None regarding dental treatment benefits.
* **Information That Cannot Be Confirmed:** Whether the policy provides a ₹2,00,000 annual dental treatment benefit, as dental benefits are not mentioned in the provided policy context.
* **Source Page and Section:** Not applicable (no section in the available policy text mentions dental treatment).

SOURCE PAGES / SECTIONS:
- Page 1 | SAMPLE HEALTH INSURANCE POLICY
- Page 2 | 8. Maternity Benefit
- Page 3 | 18. Policy Support Questions


**Claims test cases**

Test 1 — Potentially covered hospitalization

In [ ]:
claim1 = """
The insured person was admitted to a hospital for 3 days
because of a medically necessary illness.
The policy has been active for 2 years.
The hospital bill is ₹80,000.
"""

display_claim_precheck(claim1)

CLAIMS PRE-CHECK

CLAIM SCENARIO:

The insured person was admitted to a hospital for 3 days
because of a medically necessary illness.
The policy has been active for 2 years.
The hospital bill is ₹80,000.


PRELIMINARY ASSESSMENT:
### 1. Potentially Relevant Coverage
* **Hospitalization Coverage:** Medically necessary hospitalization for illness is covered when the insured person is admitted for more than 24 hours. Eligible expenses include room charges, nursing charges, doctor fees, prescribed medicines, diagnostic tests, and eligible hospital services (Page: 1, Section: SAMPLE HEALTH INSURANCE POLICY).
* **Policy Tenure/Limit:** The annual sum insured is ₹10,00,000 for geographical coverage within India (Page: 1, Section: SAMPLE HEALTH INSURANCE POLICY; Page: 2, Section: 8. Maternity Benefit).

### 2. Waiting Period Check
* **General Illness Waiting Period:** Requires a 30-day waiting period from policy start for illnesses (Page: 1, Section: SAMPLE HEALTH INSURANCE POLICY). Since the 

Test 2 — Pre-existing disease waiting period

In [ ]:
claim2 = """
The insured person was hospitalized for treatment of a
pre-existing disease. The policy has been active for
only 12 months.
"""

display_claim_precheck(claim2)

CLAIMS PRE-CHECK

CLAIM SCENARIO:

The insured person was hospitalized for treatment of a
pre-existing disease. The policy has been active for
only 12 months.


PRELIMINARY ASSESSMENT:
### 1. Potentially Relevant Coverage
* **Hospitalization Coverage:** Covers medically necessary hospitalization for illness or accidental injury when the insured person is admitted for more than 24 hours. Covered expenses may include room charges, nursing charges, doctor fees, prescribed medicines, diagnostic tests, and eligible hospital services (Page 1, Section: SAMPLE HEALTH INSURANCE POLICY).

### 2. Waiting Period Check
* **Pre-existing Diseases Waiting Period:** The policy stipulates a waiting period of 36 months for pre-existing diseases (Page 1, Section: SAMPLE HEALTH INSURANCE POLICY).
* **Scenario Applicability:** The policy has been active for only 12 months, which does not satisfy the required 36-month waiting period for pre-existing diseases (Page 1, Section: SAMPLE HEALTH INSURANCE POLICY).

Test 3 — Missing information

In [ ]:
claim3 = """
The customer says they received hospital treatment and
want to submit a claim.
"""

display_claim_precheck(claim3)

CLAIMS PRE-CHECK

CLAIM SCENARIO:

The customer says they received hospital treatment and
want to submit a claim.


PRELIMINARY ASSESSMENT:
### 1. Potentially Relevant Coverage
* **Hospitalization Reimbursement:** The insured may pay eligible hospital bills and submit required claim documents to the insurer or claims administrator for eligibility assessment according to policy terms (Page 2, Section 8. Maternity Benefit).
* **Cashless Authorization:** Available at eligible network hospitals subject to pre-authorization (Page 2, Section 8. Maternity Benefit).
* **Overall Policy Limit:** Annual sum insured up to I10,00,000 (Page 2, Section 8. Maternity Benefit).

---

### 2. Waiting Period Check
* The claims team verifies applicable waiting periods during evaluation (Page 3, Section 18. Policy Support Questions). 
* Specific waiting period terms for general hospitalization are not specified in the available policy information.

---

### 3. Limits / Sub-limits
The following sub-limits app

Test 4 — Possible exclusion

In [ ]:
claim4 = """
The customer wants reimbursement for a cosmetic procedure.
The customer has submitted a hospital bill.
"""

display_claim_precheck(claim4)

CLAIMS PRE-CHECK

CLAIM SCENARIO:

The customer wants reimbursement for a cosmetic procedure.
The customer has submitted a hospital bill.


PRELIMINARY ASSESSMENT:
### 1. Potentially Relevant Coverage
* Reimbursement claims allow the insured to pay eligible hospital bills and submit required claim documents to the insurer or authorized claims administrator for assessment according to policy terms (Page 2, Section 8. Maternity Benefit).
* Hospitalization room rent charges are covered up to I5,000 per day for standard room rent and up to I10,000 per day for ICU charges (Page 1, Section SAMPLE HEALTH INSURANCE POLICY; Page 2, Section 8. Maternity Benefit).
* However, coverage for cosmetic or aesthetic treatment is restricted and excluded unless it qualifies as eligible accident-related treatment or is specifically endorsed (Page 2, Section 8. Maternity Benefit).

### 2. Waiting Period Check
* General waiting period is 30 days from the policy start date for illnesses, except for accidental

**Build an Evaluation Section**

Create a dataset

In [67]:
evaluation_tests = [

    {
        "question": "What is the ICU limit?",
        "expected": "supported"
    },
   {
        "question": "Does the policy cover unlimited international treatment?",
        "expected": "unsupported"
    }
]

**Automatic evaluation**

In [62]:
def evaluate_rag_system(test_cases):

    results = []

    for test in test_cases:

        question = test["question"]
        expected = test["expected"]

        retrieved = retrieve_documents(
            question,
            k=5
        )

        retrieved_count = len(retrieved)

        response = ask_policy_question(
            question
        )

        answer = response["answer"]

        if expected == "supported":

            passed = (
                retrieved_count > 0
                and len(answer.strip()) > 0
            )

        else:

            refusal_phrases = [
                "cannot confirm",
                "not specified",
                "not available",
                "cannot be determined",
                "insufficient"
            ]

            answer_lower = answer.lower()

            passed = any(
                phrase in answer_lower
                for phrase in refusal_phrases
            )

        results.append({
            "question": question,
            "expected": expected,
            "retrieved_chunks": retrieved_count,
            "passed": passed
        })

    return results

In [68]:
evaluation_results = evaluate_rag_system(
    evaluation_tests
)

In [69]:
#Display answer
for result in evaluation_results:

    print("-" * 80)

    print("Question:")
    print(result["question"])

    print("Expected:")
    print(result["expected"])

    print("Retrieved chunks:")
    print(result["retrieved_chunks"])

    print("Passed:")
    print(result["passed"])

--------------------------------------------------------------------------------
Question:
What is the ICU limit?
Expected:
supported
Retrieved chunks:
5
Passed:
True
--------------------------------------------------------------------------------
Question:
Does the policy cover unlimited international treatment?
Expected:
unsupported
Retrieved chunks:
5
Passed:
True


**Calculate evaluation accuracy**

In [70]:
total_tests = len(evaluation_results)

passed_tests = sum(
    result["passed"]
    for result in evaluation_results
)

evaluation_accuracy = (
    passed_tests / total_tests
) * 100

print("Total Tests:", total_tests)
print("Passed Tests:", passed_tests)
print(
    f"Evaluation Accuracy: {evaluation_accuracy:.2f}%"
)

Total Tests: 2
Passed Tests: 2
Evaluation Accuracy: 100.00%


**Retrieval Quality Evaluation**

In [71]:
retrieval_tests = [

    {
        "question": "What is the room rent limit?",
        "expected_section": "Room Rent"
    },

    {
        "question": "What is the pre-existing disease waiting period?",
        "expected_section": "Waiting"
    },

    {
        "question": "What documents are needed for a claim?",
        "expected_section": "Documents"
    },

    {
        "question": "Is cosmetic treatment covered?",
        "expected_section": "Exclusions"
    }
]

In [72]:
def evaluate_retrieval(test_cases):

    passed = 0

    for test in test_cases:

        results = retrieve_documents(
            test["question"],
            k=5
        )

        combined_text = " ".join(
            result["text"]
            for result in results
        ).lower()

        expected_keyword = (
            test["expected_section"]
            .lower()
        )

        if expected_keyword in combined_text:
            passed += 1

        print("-" * 80)
        print("Question:", test["question"])
        print("Expected:", test["expected_section"])
        print(
            "Retrieved:",
            expected_keyword in combined_text
        )

    accuracy = (
        passed / len(test_cases)
    ) * 100

    print("\nRetrieval Test Accuracy:",
          f"{accuracy:.2f}%")

    return accuracy

In [73]:
retrieval_accuracy = evaluate_retrieval(
    retrieval_tests
)

--------------------------------------------------------------------------------
Question: What is the room rent limit?
Expected: Room Rent
Retrieved: True
--------------------------------------------------------------------------------
Question: What is the pre-existing disease waiting period?
Expected: Waiting
Retrieved: True
--------------------------------------------------------------------------------
Question: What documents are needed for a claim?
Expected: Documents
Retrieved: True
--------------------------------------------------------------------------------
Question: Is cosmetic treatment covered?
Expected: Exclusions
Retrieved: True

Retrieval Test Accuracy: 100.00%


**Architecture**

                 USER
                   │
                   ▼
          Policy Question /
           Claim Scenario
                   │
                   ▼
          Query Embedding
                   │
                   ▼
          FAISS Vector Search
                   │
                   ▼
       Similarity Threshold Filter
                   │
                   ▼
       Top Relevant Policy Chunks
                   │
             ┌─────┴─────┐
             │           │
             ▼           ▼
          Page        Section
             │           │
             └─────┬─────┘
                   │
                   ▼
             RAG Context
                   │
                   ▼
                LLM
                   │
          ┌────────┴────────┐
          │                 │
          ▼                 ▼
     Supported          Unsupported
      Question            Question
          │                 │
          ▼                 ▼
    Grounded Answer       Refusal
          │
          ▼
          Source Page + Section